# Prevención de fraudes con tarjetas de crédito
## Reconstrucción metodológica validable · 2026 — v3

**Alcance:** proyecto académico revisado, no un sistema bancario en producción. Se conservaron sin alteraciones los notebooks de 2022. Este cuaderno reúne y corrige elementos de `Copia_de_Modelo_Final_Grupo_3(1).ipynb` y `Andrés_ok(1).ipynb`, contrastados con `Informe - Grupo 3.pdf`.

**Objetivo:** comparar clasificadores para identificar transacciones etiquetadas como fraude, evitar fugas de información y obtener métricas únicamente mediante evaluación independiente.

**Reglas:** (1) separar el test original antes del EDA supervisado; (2) aprender imputaciones y balanceos dentro de cada entrenamiento de CV; (3) seleccionar modelo solo en CV; (4) seleccionar umbral exclusivamente en calibración; (5) evaluar test al final y una sola vez; (6) documentar prevalencia, distribución y limitaciones del conjunto de datos. **Los datos pueden ser sintéticos**: no inferir eficacia sobre transacciones bancarias reales.

**Distinción histórica:** las métricas del informe grupal de 2022 (cercanas al 100 %) NO son resultados de este experimento y no deben reutilizarse como evidencia predictiva independiente.

**v3 / protocolo fijado antes del test:** `FRAUD_MODE=completo`, 3 folds estratificados, 6 iteraciones de búsqueda (Average Precision), 640.000 entrenamiento, 160.000 calibración de umbral F2 y 200.000 test cerrado. No se usa la prevalencia de test en desarrollo. Random Forest paralelo en 4 hilos dentro de cada fit para agilizar la ejecución. Evaluar test una sola vez tras revisar las salidas de desarrollo.

**Nota operativa:** 3 folds y 6 iteraciones de búsqueda en 640.000 registros: elección deliberada por límite de memoria/CPU del entorno; búsqueda limitada, no exhaustiva. Paralelismo solamente dentro de Random Forest (`n_jobs=4`), no entre folds.


### Trazabilidad documental

Referencias históricas locales (no se cargan ni reescriben):

| Fuente | Función |
|---|---|
| `Copia_de_Modelo_Final_Grupo_3(1).ipynb` | EDA, árbol, Random Forest y búsqueda de parámetros. |
| `Andrés_ok(1).ipynb` | Regresión Logística, particiones estratificadas y F2. |
| `Informe - Grupo 3.pdf` | Informe grupal histórico, resultados y decisiones de 2022. |

Bibliografía especializada: **Kuhn y Johnson (2013), *Applied Predictive Modeling*, caps. 4, 11 y 16**; **Fernández et al. (2018), *Learning from Imbalanced Data Sets*, caps. 3–5**; **Kuhn y Johnson (2019/2020), *Feature Engineering and Selection*, cap. 3**; **Provost y Fawcett (2013), *Data Science for Business*, cap. 7**.

## 0. Configuración

Este cuaderno ofrece dos modos. **`exploracion`** usa un máximo de 60.000 observaciones del entrenamiento para comprobar el flujo y deja el conjunto de prueba **cerrado**. **`completo`** utiliza todos los registros del entrenamiento e incorpora una búsqueda más amplia. La evaluación final requiere cambiar *ambos* `MODO = 'completo'` y `EVALUAR_TEST_FINAL = True` **antes de empezar la ejecución**, y no volver a utilizar el test para ajustar modelos.

El CSV puede ubicarse junto al notebook, en `data/card_transdata.csv`, o indicarse mediante `FRAUD_CSV_PATH`. No se descarga ningún dataset automáticamente. Requisitos: Python, numpy, pandas, matplotlib, scikit-learn, imbalanced-learn e IPython.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.metadata, json, os, platform, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, fbeta_score,
    precision_score, recall_score, precision_recall_curve,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay, make_scorer
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler

SEED = 42
MODO = os.environ.get('FRAUD_MODE', 'exploracion')  # 'exploracion' o 'completo'
EVALUAR_TEST_FINAL = os.environ.get('FRAUD_FINAL', '0') == '1'  # cambio deliberado
INCLUIR_SMOTENC = os.environ.get('FRAUD_SMOTENC', '0') == '1'
FOLDS = int(os.environ.get('FRAUD_FOLDS', '3' if MODO == 'exploracion' else '5'))
ITERACIONES_BUSQUEDA = int(os.environ.get('FRAUD_SEARCH_ITER', '3' if MODO == 'exploracion' else '12'))
MAX_FILAS_EXPLORACION = 60_000

assert MODO in ('exploracion', 'completo')
assert FOLDS >= 2 and ITERACIONES_BUSQUEDA >= 1
if EVALUAR_TEST_FINAL and MODO != 'completo':
    raise ValueError('La evaluación final exige MODO="completo": evitar usar el test durante exploración.')
VERSIONES = {
    nombre: importlib.metadata.version(nombre)
    for nombre in ('numpy', 'pandas', 'scikit-learn', 'imbalanced-learn', 'matplotlib')
}
print('Modo:', MODO, '| Test final habilitado:', EVALUAR_TEST_FINAL)
print('Python:', sys.version.split()[0], '| Versiones:', VERSIONES)

## 1. Carga, verificación del esquema y huella del CSV

Comprobaciones: variables esperadas, variable objetivo binaria, variables indicadoras 0/1, infinitos, duplicados y faltantes. **No eliminar observaciones ni imputar valores utilizando el dataset completo.** La prevalencia global se documenta para justificar la partición estratificada, sin usar el conjunto test para elegir predictores.

In [ ]:
ruta_env = os.environ.get('FRAUD_CSV_PATH', '').strip()
rutas = ([Path(ruta_env).expanduser()] if ruta_env else []) + [
    Path('data/card_transdata.csv'), Path('card_transdata.csv'),
    Path('../data/card_transdata.csv'), Path('/mnt/data/card_transdata.csv'),
]
RUTA_CSV = next((r for r in rutas if r.is_file()), None)
if RUTA_CSV is None:
    raise FileNotFoundError(
        'No encuentro card_transdata.csv. Copialo a data/card_transdata.csv, '
        'al directorio del notebook o definí FRAUD_CSV_PATH. No uses otro archivo sin verificar su origen.'
    )

def sha256_archivo(ruta, bloque=2**20):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for parte in iter(lambda: f.read(bloque), b''):
            h.update(parte)
    return h.hexdigest()

HUELLA_CSV = sha256_archivo(RUTA_CSV)
TARGET = 'fraud'
CONTINUAS = ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price']
BINARIAS = ['repeat_retailer', 'used_chip', 'used_pin_number', 'online_order']
FEATURES = CONTINUAS + BINARIAS  # Mantener repeat_retailer salvo que una ablation CV justifique excluirla.
COLUMNAS = FEATURES + [TARGET]
base = pd.read_csv(RUTA_CSV, low_memory=False)
assert base.columns.is_unique, 'Columnas duplicadas: investigar.'
assert set(COLUMNAS).issubset(base.columns), f'Faltan: {set(COLUMNAS)-set(base.columns)}'
if set(base.columns) != set(COLUMNAS):
    print('Advertencia: columnas adicionales no utilizadas:', sorted(set(base.columns)-set(COLUMNAS)))
base = base[COLUMNAS].copy()
for c in COLUMNAS:
    base[c] = pd.to_numeric(base[c], errors='raise')
assert base[TARGET].notna().all(), 'Objetivo con faltantes: investigar antes de modelar.'
assert base[TARGET].isin([0, 1]).all() and base[TARGET].nunique() == 2
for c in BINARIAS:
    assert base.loc[base[c].notna(), c].isin([0, 1]).all(), f'{c} no es indicador 0/1.'
assert np.isfinite(base[FEATURES].dropna().to_numpy(dtype=float)).all(), 'Hay infinitos.'
print('CSV:', RUTA_CSV.resolve())
print('SHA256:', HUELLA_CSV)
print('Filas/columnas:', base.shape)
duplicados_exactos = int(base.duplicated().sum())
print('Duplicados exactos:', duplicados_exactos)
if duplicados_exactos:
    raise ValueError(
        f'El CSV tiene {duplicados_exactos} filas duplicadas exactas. '
        'Inspeccioná su significado antes del split: registros duplicados pueden contaminar test.'
    )
print('Faltantes:', base.isna().sum().to_dict())
print('Frecuencia objetivo:', base[TARGET].value_counts().sort_index().to_dict())
print('Prevalencia original:', f'{base[TARGET].mean():.4%}')

## 2. Partición estratificada: entrenamiento / calibración / prueba

Primero se reserva **20 % para test**. El 80 % restante se divide en **entrenamiento del modelo (80 %)** y **calibración del umbral (20 %)**. En consecuencia, en modo completo las fracciones del dataset son ~64 % entrenamiento, ~16 % calibración y ~20 % prueba.

La calibración se usa **solo para seleccionar el umbral de decisión** después de seleccionar el modelo y sus hiperparámetros mediante CV sobre entrenamiento. El conjunto de prueba final no interviene en ninguna de estas decisiones.

In [ ]:
X = base[FEATURES].copy()
y = base[TARGET].astype(int).copy()
assert TARGET not in X.columns
X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_fit_all, X_cal, y_fit_all, y_cal = train_test_split(
    X_train_pool, y_train_pool, test_size=0.20,
    stratify=y_train_pool, random_state=SEED + 1
)
if MODO == 'exploracion' and len(X_fit_all) > MAX_FILAS_EXPLORACION:
    X_fit, _, y_fit, _ = train_test_split(
        X_fit_all, y_fit_all, train_size=MAX_FILAS_EXPLORACION,
        stratify=y_fit_all, random_state=SEED + 2
    )
else:
    X_fit, y_fit = X_fit_all, y_fit_all
for a, b in [(X_fit_all, X_cal), (X_fit_all, X_test), (X_cal, X_test)]:
    assert a.index.intersection(b.index).empty
assert all(0 < y_.mean() < 1 for y_ in (y_fit, y_cal, y_test))
print(pd.DataFrame([
    {'conjunto': 'Entrenamiento disponible', 'n': len(X_fit_all), 'fraude_pct': 100*y_fit_all.mean()},
    {'conjunto': 'Entrenamiento utilizado', 'n': len(X_fit), 'fraude_pct': 100*y_fit.mean()},
    {'conjunto': 'Calibración del umbral', 'n': len(X_cal), 'fraude_pct': 100*y_cal.mean()},
    {'conjunto': 'Test FINAL (cerrado)', 'n': len(X_test), 'fraude_pct': np.nan},
]).round(3).to_string(index=False))
# La prevalencia de test deliberadamente no se imprime antes de la evaluación final.
# Evitar inspeccionar distribuciones condicionadas de la clase objetivo en test.
# No explorar X_test/y_test antes de la evaluación final.

## 3. EDA exclusivamente sobre entrenamiento

Esta sección reproduce los principales análisis del grupo (frecuencias, distribuciones, asociaciones). No toma decisiones irreversibles de variables basadas en la totalidad de la base. El EDA no valida causalidad ni descarta automáticamente características por correlación bivariada.

In [ ]:
df_eda = X_fit.copy()
df_eda[TARGET] = y_fit
print('EDA sobre entrenamiento, n:', len(df_eda))
display(pd.DataFrame({
    'dtype': df_eda.dtypes.astype(str),
    'nulos': df_eda.isna().sum(),
    'unicos': df_eda.nunique(dropna=True)
}))
display(df_eda[CONTINUAS].describe(percentiles=[.05, .25, .5, .75, .95]).T)
display(df_eda[FEATURES + [TARGET]].corr(method='spearman').round(3))
fig, ax = plt.subplots(figsize=(5.5, 3))
conteos = y_fit.value_counts().sort_index()
ax.bar(['Legítimas', 'Fraude'], [int(conteos.get(0, 0)), int(conteos.get(1, 0))])
ax.set(ylabel='N.º de operaciones', title='Clase objetivo — entrenamiento')
plt.tight_layout(); plt.show()
for col in CONTINUAS:
    fig, ax = plt.subplots(figsize=(6.5, 3))
    for val, titulo in [(0, 'Legítima'), (1, 'Fraude')]:
        valores = df_eda.loc[df_eda[TARGET].eq(val), col].dropna()
        if (valores < 0).any():
            ax.hist(valores, bins=35, density=True, histtype='step', label=titulo)
            etiqueta = col
        else:
            ax.hist(np.log1p(valores), bins=35, density=True, histtype='step', label=titulo)
            etiqueta = f'log1p({col})'
    ax.set(xlabel=etiqueta, ylabel='Densidad', title=f'Distribución — {col}')
    ax.legend(); plt.tight_layout(); plt.show()
for col in BINARIAS:
    print('\n', col)
    display(pd.crosstab(df_eda[col], df_eda[TARGET], normalize='index').round(3))

## 4. Preprocesamiento dentro de cada fold y modelos candidatos

- `DummyClassifier` establece el baseline.
- Regresión Logística: imputación + escalado robusto dentro de CV y ponderación de clases.
- Decision Tree y Random Forest: imputación dentro de CV, sin escalado innecesario.
- Submuestreo y SMOTENC (opcional) se ejecutan **dentro de cada fold** a través de `imblearn.pipeline.Pipeline`.
- Se conservan siete predictores, incluida `repeat_retailer`; una posible eliminación sería un experimento de ablation independiente.

**Decisión principal:** seleccionar por **Average Precision**; informar también ROC-AUC, F2, precision y recall a umbral 0,5 solo como diagnósticos. La optimización del umbral se hará después, sobre un conjunto diferente.

In [ ]:
prep_escalada = ColumnTransformer([
    ('continuas', SKPipeline([
        ('imputar', SimpleImputer(strategy='median')),
        ('escalar', RobustScaler())
    ]), CONTINUAS),
    ('binarias', SimpleImputer(strategy='most_frequent'), BINARIAS)
], remainder='drop')
prep_arbol = ColumnTransformer([
    ('continuas', SimpleImputer(strategy='median'), CONTINUAS),
    ('binarias', SimpleImputer(strategy='most_frequent'), BINARIAS)
], remainder='drop')

def pipeline_modelo(modelo, *, escala=False, sampler=None):
    pasos = [('preprocesamiento', clone(prep_escalada if escala else prep_arbol))]
    if sampler is not None:
        pasos.append(('remuestreo', sampler))
    pasos.append(('modelo', modelo))
    return ImbPipeline(pasos) if sampler is not None else SKPipeline(pasos)

N_ARBOLES = 30 if MODO == 'exploracion' else 100
modelos = {
    'Baseline (prior)': pipeline_modelo(DummyClassifier(strategy='prior')),
    'Regresión Logística ponderada': pipeline_modelo(
        LogisticRegression(class_weight='balanced', max_iter=1500, random_state=SEED), escala=True
    ),
    'Árbol de decisión': pipeline_modelo(
        DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=SEED)
    ),
    'Random Forest original': pipeline_modelo(
        RandomForestClassifier(n_estimators=N_ARBOLES, min_samples_leaf=3,
                               max_depth=18, n_jobs=4, random_state=SEED)
    ),
    'Random Forest ponderado': pipeline_modelo(
        RandomForestClassifier(n_estimators=N_ARBOLES, min_samples_leaf=3,
                               max_depth=18, class_weight='balanced', n_jobs=4, random_state=SEED)
    ),
    'Random Forest + submuestreo': pipeline_modelo(
        RandomForestClassifier(n_estimators=N_ARBOLES, min_samples_leaf=3,
                               max_depth=18, n_jobs=4, random_state=SEED),
        sampler=RandomUnderSampler(random_state=SEED)
    ),
}
if INCLUIR_SMOTENC:
    # Salida preprocesador: continuas [0, 1, 2], indicadoras [3, 4, 5, 6].
    modelos['Random Forest + SMOTENC'] = pipeline_modelo(
        RandomForestClassifier(n_estimators=N_ARBOLES, min_samples_leaf=3,
                               max_depth=18, n_jobs=4, random_state=SEED),
        sampler=SMOTENC(categorical_features=[3, 4, 5, 6], k_neighbors=5, random_state=SEED)
    )
cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
metricas_cv = {
    'ap': 'average_precision',
    'roc': 'roc_auc',
    'f2_05': make_scorer(fbeta_score, beta=2, zero_division=0),
    'precision_05': make_scorer(precision_score, zero_division=0),
    'recall_05': make_scorer(recall_score, zero_division=0),
}
print('Modelos candidatos:', list(modelos), '| folds:', FOLDS)

## 5. Comparación homogénea mediante validación cruzada

Todos los clasificadores utilizan **exactamente los mismos folds**. El valor medio de CV es una herramienta de selección, **no** la estimación final para comunicar al portfolio. No se consulta test aquí.

In [ ]:
registros_cv = []
for nombre, estimador in modelos.items():
    print('Validando:', nombre)
    resultados = cross_validate(
        estimador, X_fit, y_fit, cv=cv, scoring=metricas_cv,
        n_jobs=1, return_train_score=False, error_score='raise'
    )
    registros_cv.append({
        'modelo': nombre,
        'ap_media': float(resultados['test_ap'].mean()),
        'ap_desv': float(resultados['test_ap'].std()),
        'roc_auc_media': float(resultados['test_roc'].mean()),
        'f2_05_media': float(resultados['test_f2_05'].mean()),
        'precision_05_media': float(resultados['test_precision_05'].mean()),
        'recall_05_media': float(resultados['test_recall_05'].mean()),
    })
tabla_cv = pd.DataFrame(registros_cv).sort_values('ap_media', ascending=False).reset_index(drop=True)
display(tabla_cv.round(4))
nombre_basico = str(tabla_cv.iloc[0]['modelo'])
modelo_basico = clone(modelos[nombre_basico])
print('Líder de CV (aún NO seleccionado definitivamente):', nombre_basico)

## 6. Optimización de Random Forest sin utilizar test ni calibración

`RandomizedSearchCV` explora tanto la variante ponderada como la no ponderada. La métrica es **Average Precision**, no accuracy. La búsqueda ocurre únicamente sobre `X_fit/y_fit`. Su mejor puntuación está sujeta al **sesgo de selección** y se utiliza solo para elegir; no se publica como desempeño final.

Para reducir cómputo el modo exploración usa pocas iteraciones. El modo completo examina más configuraciones, pero **no convierte esta búsqueda en exhaustiva**.

In [ ]:
busqueda = RandomizedSearchCV(
    estimator=pipeline_modelo(RandomForestClassifier(random_state=SEED, n_jobs=4)),
    param_distributions={
        'modelo__n_estimators': [50, 100, 150],
        'modelo__max_depth': [10, 16, 22, None],
        'modelo__min_samples_leaf': [1, 3, 8],
        'modelo__max_features': ['sqrt', 0.75],
        'modelo__class_weight': [None, 'balanced'],
    },
    n_iter=ITERACIONES_BUSQUEDA, cv=cv, scoring='average_precision',
    refit=True, random_state=SEED, n_jobs=1, error_score='raise'
)
busqueda.fit(X_fit, y_fit)
print('Mejor AP de la búsqueda (CV de desarrollo):', round(float(busqueda.best_score_), 5))
print('Parámetros:', busqueda.best_params_)
# Selección interna con AP. Esta comparación tiene optimismo por búsqueda;
# el conjunto test permanece intacto para obtener una estimación independiente.
if busqueda.best_score_ > float(tabla_cv.iloc[0]['ap_media']):
    nombre_elegido = 'Random Forest optimizado'
    estimador_elegido = clone(busqueda.best_estimator_)
else:
    nombre_elegido = nombre_basico
    estimador_elegido = clone(modelo_basico)
print('Modelo seleccionado:', nombre_elegido)

## 7. Entrenamiento elegido y selección del umbral en calibración independiente

El estimador seleccionado se entrena **una sola vez sobre `X_fit`** (sin utilizar calibración ni test). Después se predicen probabilidades sobre `X_cal` y se selecciona el umbral que maximiza F2. **Ese F2 es una métrica de selección, no de rendimiento final**.

El umbral de máxima F2 puede resultar inadecuado para un negocio real: sin costos de fraude y de falsas alertas, no se establece aquí una política operativa de bloqueo de transacciones. No se reentrena después de seleccionar el umbral para evitar inconsistencias en los puntajes calibrados.

In [ ]:
modelo_final = clone(estimador_elegido)
modelo_final.fit(X_fit, y_fit)
score_cal = modelo_final.predict_proba(X_cal)[:, 1]
assert np.isfinite(score_cal).all() and ((score_cal >= 0) & (score_cal <= 1)).all()
precision_cal, recall_cal, thresholds_cal = precision_recall_curve(y_cal, score_cal)
b2 = 4.0
f2_candidates = (5 * precision_cal[:-1] * recall_cal[:-1]) / np.maximum(
    4 * precision_cal[:-1] + recall_cal[:-1], 1e-12
)
indice_mejor = int(np.argmax(f2_candidates))
umbral_final = float(thresholds_cal[indice_mejor])
cal_pred = (score_cal >= umbral_final).astype(int)
print('Umbral elegido en calibración:', round(umbral_final, 6))
print('F2 sobre calibración (NO test):', round(fbeta_score(y_cal, cal_pred, beta=2, zero_division=0), 5))
print('Average Precision de calibración:', round(average_precision_score(y_cal, score_cal), 5))
print('Precision / recall calibración:',
      round(precision_score(y_cal, cal_pred, zero_division=0), 5),
      round(recall_score(y_cal, cal_pred, zero_division=0), 5))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(recall_cal, precision_cal)
ax.set(xlabel='Recall', ylabel='Precision', title='Precision–Recall — CALIBRACIÓN (no test)')
plt.tight_layout(); plt.show()

## 7b. Congelación del modelo antes de consultar test

Se guardan el modelo ajustado, el umbral elegido y el manifiesto de las decisiones tomadas **sin usar la variable objetivo de test**. Este artefacto se utilizará para una única evaluación final, en ejecución separada.

In [ ]:
import joblib
DIRECTORIO_PRETEST = Path('outputs/prevencion_fraudes_revision_2026')
DIRECTORIO_PRETEST.mkdir(parents=True, exist_ok=True)
RUTA_MODELO_PRETEST = DIRECTORIO_PRETEST / 'modelo_congelado_pretest.joblib'
RUTA_MANIFIESTO = DIRECTORIO_PRETEST / 'manifiesto_congelado_pretest.json'
if RUTA_MANIFIESTO.exists():
    raise FileExistsError('Ya hay un manifiesto de congelación. No sobrescribirlo sin respaldo y aprobación.')
joblib.dump(modelo_final, RUTA_MODELO_PRETEST, compress=3)
MANIFIESTO = {
    'tipo': 'REVISION_METODOLOGICA_2026', 'estado': 'MODELO_CONGELADO_PRETEST',
    'csv_sha256': HUELLA_CSV, 'modelo': nombre_elegido,
    'modelo_joblib_sha256': sha256_archivo(RUTA_MODELO_PRETEST),
    'n_entrenamiento': int(len(X_fit)), 'n_calibracion': int(len(X_cal)),
    'n_prueba_reservada': int(len(X_test)),
    'test_indices_sha256': hashlib.sha256(np.asarray(X_test.index, dtype=np.int64).tobytes()).hexdigest(),
    'umbral_f2_calibrado': float(umbral_final),
    'f2_calibracion': float(fbeta_score(y_cal,cal_pred,beta=2,zero_division=0)),
    'ap_calibracion': float(average_precision_score(y_cal,score_cal)),
    'cv_folds': int(FOLDS), 'search_iter': int(ITERACIONES_BUSQUEDA),
    'mejor_ap_search_cv': float(busqueda.best_score_),
    'parametros_search': busqueda.best_params_,
    'cv_modelos': tabla_cv.to_dict(orient='records'),
    'semilla': int(SEED), 'versiones': VERSIONES,
    'datetime_utc': datetime.now(timezone.utc).isoformat(),
}
RUTA_MANIFIESTO.write_text(json.dumps(MANIFIESTO,ensure_ascii=False,indent=2),encoding='utf-8')
print('Modelo y manifiesto congelados sin evaluar test:', RUTA_MODELO_PRETEST, RUTA_MANIFIESTO)


## 8. Evaluación final — **cerrada por defecto**

Habilitar únicamente tras revisar las decisiones anteriores y disponer del dataset original correcto. En `MODO='exploracion'`, esta sección **no calcula métricas de test**. En `MODO='completo'`, solo procede si `EVALUAR_TEST_FINAL=True` se estableció **antes de iniciar la ejecución**.

No utilizar el test para cambiar el modelo, el balanceo, los hiperparámetros o el umbral después de ver sus resultados. Una evaluación repetida por ajustes sucesivos deja de ser independiente.

In [ ]:
if not EVALUAR_TEST_FINAL:
    print('TEST FINAL CERRADO. No se calculan métricas finales en esta ejecución.')
    metricas_finales = None
else:
    assert MODO == 'completo' and len(X_fit) == len(X_fit_all), (
        'El test exige entrenamiento completo.'
    )
    score_test = modelo_final.predict_proba(X_test)[:, 1]
    pred_test = (score_test >= umbral_final).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_test, labels=[0, 1]).ravel()
    metricas_finales = {
        'tipo': 'REVISION_2026_NO_TESIS_2022',
        'dataset_sha256': HUELLA_CSV,
        'modelo': nombre_elegido,
        'n_entrenamiento': int(len(X_fit)),
        'n_calibracion': int(len(X_cal)),
        'n_prueba': int(len(X_test)),
        'fraude_pct_prueba': float(100*y_test.mean()),
        'umbral_calibrado': umbral_final,
        'accuracy': float(accuracy_score(y_test, pred_test)),
        'precision': float(precision_score(y_test, pred_test, zero_division=0)),
        'recall': float(recall_score(y_test, pred_test, zero_division=0)),
        'f1': float(f1_score(y_test, pred_test, zero_division=0)),
        'f2': float(fbeta_score(y_test, pred_test, beta=2, zero_division=0)),
        'average_precision': float(average_precision_score(y_test, score_test)),
        'roc_auc': float(roc_auc_score(y_test, score_test)),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'versions': VERSIONES,
    }
    display(pd.DataFrame([metricas_finales]).T.rename(columns={0: 'Valor'}))
    print('\nReporte de clasificación:\n', classification_report(y_test, pred_test, zero_division=0))
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, pred_test, labels=[0, 1], ax=ax)
    ax.set_title('Matriz de confusión — TEST FINAL'); plt.tight_layout(); plt.show()
    fig, ax = plt.subplots(figsize=(6, 4))
    PrecisionRecallDisplay.from_predictions(y_test, score_test, ax=ax)
    ax.set_title('PR — TEST FINAL'); plt.tight_layout(); plt.show()
    fig, ax = plt.subplots(figsize=(6, 4))
    RocCurveDisplay.from_predictions(y_test, score_test, ax=ax)
    ax.set_title('ROC — TEST FINAL'); plt.tight_layout(); plt.show()
    dir_salida = Path('outputs') / 'prevencion_fraudes_revision_2026'
    dir_salida.mkdir(parents=True, exist_ok=True)
    archivo_resultado = dir_salida / 'metricas_prueba_final.json'
    archivo_resultado.write_text(
        json.dumps(metricas_finales, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print('Reporte de métricas guardado en:', archivo_resultado)

## 9. Interpretación y alcance de resultados

**Qué demuestra:** diseño de un flujo reproducible, preprocesamiento aislado, comparación homogénea, búsqueda de hiperparámetros, selección de umbral y evaluación separada.

**Qué no demuestra automáticamente:** rendimiento sobre transacciones bancarias reales, eficacia en producción, calibración probabilística, robustez temporal, generalización entre clientes/tarjetas, ausencia de sesgos o reducción monetaria de fraudes. El dataset no aporta identificadores de cliente ni marcas temporales para establecer una validación temporal o por grupos. **No presentar 99–100 % de accuracy del informe histórico como desempeño validado.**

**Calibración:** aquí el término se refiere a *selección de umbral en un conjunto reservado*. No se realizó calibración probabilística (`CalibratedClassifierCV`) ni una auditoría de confiabilidad de probabilidades. Si se pretende utilizar probabilidades como riesgos absolutos, incorporar esa etapa por separado sin utilizar test.

**Matriz de confusión (filas=verdad, columnas=predicción):** `[[TN, FP], [FN, TP]]`. Los falsos negativos son fraudes que se clasifican como legítimos; los falsos positivos son legítimas que se clasifican como fraude. En términos operativos, la elección de umbral debería considerar costos reales de ambos tipos de error.

**Evidencia histórica:** el informe del Grupo 03 documenta que se aplicó SMOTE antes de la partición 80/20; la prevalencia artificial del test histórico limita sus métricas. El cuaderno presente se identifica explícitamente como **revisión 2026**, distinta del trabajo académico original.

## 10. Lista de controles metodológicos

| Control | Estado del código | Alcance |
|---|---|---|
| Objetivo fuera de predictores | Implementado | `fraud` excluido explícitamente de `X`. |
| Conjunto test independiente | Implementado | Particionado antes del EDA supervisado y balanceo. |
| Umbral no ajustado en test | Implementado | Calibración separada de CV y test. |
| Preprocesamiento dentro de CV | Implementado | Pipelines con imputación y escalado según algoritmo. |
| Remuestreo dentro de CV | Implementado | `imblearn.pipeline` para submuestreo/SMOTENC. |
| Variables binarias en SMOTE | Implementado | Si se activa: SMOTENC con máscara de indicadores. |
| Optimización según Average Precision | Implementado | Búsqueda aleatoria explícita, no accuracy. |
| Modelos comparados sobre los mismos folds | Implementado | StratifiedKFold compartido. |
| Métricas finales verificables | **Pendiente de ejecución real** | Requiere CSV original y test final deliberado. |
| Validación temporal/por cliente | **No factible con columnas disponibles** | No inferir generalización operacional. |
| Costos económicos y calibración probabilística | **No realizado** | Requiere datos/objetivos adicionales. |

**Advertencia:** ninguna evaluación metodológica garantiza rendimiento; el protocolo se valida definitivamente solo después de ejecutar sobre el dataset original y revisar sus resultados y posibles particularidades.